# Language Detection

---


**Dataset:**
- [papluca/language-identification](https://huggingface.co/datasets/papluca/language-identification)


### Import used libraries


In [1]:
import pandas as pd

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", 500)

### Load Dataset


In [2]:
from datasets import load_dataset

dataset = load_dataset("papluca/language-identification")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/12.0M [00:00<?, ?B/s]

valid.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/70000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [3]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['labels', 'text'],
        num_rows: 70000
    })
    validation: Dataset({
        features: ['labels', 'text'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['labels', 'text'],
        num_rows: 10000
    })
})


In [4]:
train_df = dataset["train"].to_pandas()
val_df   = dataset["validation"].to_pandas()
test_df  = dataset["test"].to_pandas()

print("Num classes:", train_df["labels"].nunique())

Num classes: 20


In [5]:
train_df.head(10)

,labels,text
0,pt,"os chefes de defesa da estónia, letónia, lituânia, alemanha, itália, espanha e eslováquia assinarão o acordo para fornecer pessoal e financiamento para o centro."
1,bg,размерът на хоризонталната мрежа може да бъде по реда на няколко километра ( km ) за на симулация до около 100 km за на симулация .
2,zh,很好，以前从不去评价，不知道浪费了多少积分，现在知道积分可以换钱，就要好好评价了，后来我就把这段话复制走了，既能赚积分，还省事，走到哪复制到哪，最重要的是，不用认真的评论了，不用想还差多少字，直接发出就可以了，推荐给大家！！
3,th,สำหรับ ของเก่า ที่ จริงจัง ลอง honeychurch ของเก่า ที่ ไม่ 29 สำหรับ เฟอร์นิเจอร์ และ เงิน ไท ร้อง บริษัท ที่ 122 สำหรับ ลาย คราม
4,ru,Он увеличил давление .
5,pl,"S Jak sobie życzysz: Widzisz, jak Hitler zabija Żydów?"
6,ur,اس کے بارے میں ، سفید شادی کی شرح کے بعد سفید اورنمایاں طور پر سفید اورنمایاں طور پر .
7,sw,"Zabuni ya ushindani pia imekuwa rahisi kwa sifa ya kurudi kwenye mapendekezo yake ya grant , na dudovitz hivi karibuni ilikuwa kulea kwa ajili ya kuendeleza nyenzo za matangazo ya slick na kushiriki fedha kwa mipango nyingine ya mapendeleo ya umma kwa juhudi za shirikishi za afya , utetezi wa sera , homelessness , nyumbani msaada wa vurugu , msaada wa kibinafsi na maendeleo ya teknolojia ."
8,tr,"Devasa 12 yüzyıl abbatiale saint-Pierre-Et-Saint-Paul , Aziz Peter ' ın 17 yüzyılda Roma ' da tamamlama kadar Hıristiyan ' en büyük kilisesi idi ."
9,ur,موجودہ اثاثوں میں سے ایک کا اضافہ ہو سکتا ہے ۔


In [6]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   labels  70000 non-null  object
 1   text    70000 non-null  object
dtypes: object(2)
memory usage: 1.1+ MB


### EDA on training data


#### Check for NaNs


In [7]:
print("Null values:")
print(train_df.isnull().sum())

Null values:
labels    0
text      0
dtype: int64


#### Check for duplicates


In [8]:
print("Number of duplicate rows:", train_df.duplicated().sum())

Number of duplicate rows: 1020


In [9]:
duplicate_rows = train_df[train_df.duplicated(keep=False)].sort_values("text")
duplicate_rows.head(20)

,labels,text
13778,ar,- ولدى المجلس حاليا مشروع نشط لمعالجة معايير الموارد الطبيعية .
28542,ar,- ولدى المجلس حاليا مشروع نشط لمعالجة معايير الموارد الطبيعية .
44424,hi,1822 में शहर की अंग ् रेजी कॉलोनी के वित ् त-पोषण के लिए निवासियों और आगंतुकों की एक तरह की सैर की गई है .
24676,hi,1822 में शहर की अंग ् रेजी कॉलोनी के वित ् त-पोषण के लिए निवासियों और आगंतुकों की एक तरह की सैर की गई है .
1686,tr,"2001 ' in ikinci yarısında 183,000 ' den fazla kişinin görev yaptığını belirttiler ."
32904,tr,"2001 ' in ikinci yarısında 183,000 ' den fazla kişinin görev yaptığını belirttiler ."
8775,nl,"49 doden, 148 gewonden bij gewelddadige aanvallen in Irak..."
35261,nl,"49 doden, 148 gewonden bij gewelddadige aanvallen in Irak..."
35372,it,"49 morti, 148 feriti in violenti attacchi in Iraq"
40619,it,"49 morti, 148 feriti in violenti attacchi in Iraq"


In [10]:
train_df = train_df.drop_duplicates().reset_index(drop=True)

In [11]:
print("Number of duplicate rows:", train_df.duplicated().sum())

Number of duplicate rows: 0


In [12]:
print("Dataset shape:", train_df.shape)

Dataset shape: (68980, 2)


#### Check dataset balancing


In [13]:
def check_df_balancing(df: pd.DataFrame, target: str) -> pd.DataFrame:
    return pd.concat(
        [
            df[target].value_counts(),
            df[target].value_counts(normalize=True) * 100,
        ],
        axis=1, keys=["Count", "Percentage"]
    )

In [14]:
check_df_balancing(train_df, target="labels")

,Count,Percentage
labels,,
ja,3500,5.073934
de,3499,5.072485
en,3499,5.072485
es,3498,5.071035
fr,3495,5.066686
zh,3491,5.060887
bg,3464,5.021745
tr,3464,5.021745
vi,3464,5.021745


#### Sample text data for inspection

show a representative sample of data texts to find out required preprocessing steps


In [15]:
def show_samples(df: pd.DataFrame, target: str, text_col: str, n: int = 5):
    for cls in df[target].unique():
        print(f"\nClass: {cls}")
        print("-" * 40)

        sample = df[df[target] == cls].sample(n=min(n, len(df[df[target] == cls])))

        for _, row in sample.iterrows():
            print(f"{row[target]} | {row[text_col]}\n")

In [16]:
show_samples(train_df, target="labels", text_col="text", n=5)


Class: pt
----------------------------------------
pt | Um homem está a cortar uma corda com uma espada.

pt | Olivia Colman ganha segundo Prémio BAFTA

pt | Futuros KLCI negociados mistos a meio do dia

pt | Os clientes surdos processam a Starbucks, dizem que são gozados

pt | Ataque suicida do Iémen mata 7 soldados: oficiais


Class: bg
----------------------------------------
bg | fda се позова на раздели 502 , 510 , 518 , 519 , 520,701 , 704 и 903 от федералния закон за храните , наркотиците и козметичните продукти ( 21 г . ) .

bg | точно като пролетен лук .

bg | основното пристанище , kamares , лежи на 5 км ( 3 км ) от столицата аполония , и сестра му град artemon ( и двете са наречени за брат и сестра бог и богиня на древната гръцка митология ) .

bg | прочетох нещо във вестника. да , днес прочетох нещо във вестника , за което говорят , или е било списание " парад " вчера и казаха , че знаеш кога е последният епизод и какво ще се случи , а те казват , че вътрешни хора казват :

**Cleaning and Preprocessing are:**

- Lowercase
- Removing URLs
- Removing Mentions
- Handling Hashtags
- Removing extra spaces

### Cleaning and Preprocessing


In [17]:
!pip install -q emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 16.2 MB/s eta 0:00:00


In [18]:
import re
import emoji

def clean_text(text):
    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http[s]?://\S+|www\.\S+", "", text)

    # Remove mentions
    text = re.sub(r"@\w+", "", text)

    # handle hashtags
    text = re.sub(r"#(\w+)", r"\1", text)

    # Remove emoji
    text = emoji.replace_emoji(text, replace="")

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [19]:
X_train, y_train = train_df["text"], train_df["labels"]
X_val, y_val = val_df["text"], val_df["labels"]
X_test, y_test = test_df["text"], test_df["labels"]

### Model Pipeline


In [20]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

pipeline = Pipeline([
    ("vectorizing", TfidfVectorizer(
        preprocessor=clean_text,
    )),
    ("model", LinearSVC(random_state=42))
])

In [21]:
param_grid = {
    "vectorizing__analyzer": ["word", "char_wb"],
    "vectorizing__ngram_range": [(2, 3), (2, 4), (3, 5)],
    "vectorizing__max_features": [3000, 5000, 10000, None],
    "model__C": [0.01, 0.1, 1, 3, 5, 10, 20],
}

In [22]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

random_search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=10,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1,
    random_state=42
)

In [23]:
random_search.fit(X_train, y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


RandomizedSearchCV(cv=StratifiedKFold(n_splits=3, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('vectorizing',
                                              TfidfVectorizer(preprocessor=<function clean_text at 0x7f6af1d45620>)),
                                             ('model',
                                              LinearSVC(random_state=42))]),
                   n_jobs=-1,
                   param_distributions={'model__C': [0.01, 0.1, 1, 3, 5, 10,
                                                     20],
                                        'vectorizing__analyzer': ['word',
                                                                  'char_wb'],
                                        'vectorizing__max_features': [3000,
                                                                      5000,
                                                                      10000,
                                                                      None],
                                        'vectorizing__ngram_range': [(2, 3),
                                                                     (2, 4),
                                                                     (3, 5)]},
                   random_state=42, scoring='f1_macro', verbose=1)

### Best Model

In [24]:
best_model = random_search.best_estimator_

print("Best Params:")
print(random_search.best_params_)

Best Params:
{'vectorizing__ngram_range': (2, 4), 'vectorizing__max_features': None, 'vectorizing__analyzer': 'char_wb', 'model__C': 10}


### Evaluation


In [25]:
from sklearn.metrics import classification_report, f1_score, accuracy_score


y_val_pred = best_model.predict(X_val)

print("Val Accuracy:", accuracy_score(y_val, y_val_pred))
print("Val F1 Macro:", f1_score(y_val, y_val_pred, average="macro"))

print("\nClassification Report:\n")
print(classification_report(y_val, y_val_pred))

Val Accuracy: 0.9954
Val F1 Macro: 0.9953929511864207

Classification Report:

              precision    recall  f1-score   support

          ar       1.00      0.99      1.00       500
          bg       1.00      1.00      1.00       500
          de       1.00      1.00      1.00       500
          el       1.00      1.00      1.00       500
          en       0.99      1.00      1.00       500
          es       1.00      1.00      1.00       500
          fr       1.00      1.00      1.00       500
          hi       1.00      0.95      0.98       500
          it       1.00      1.00      1.00       500
          ja       1.00      1.00      1.00       500
          nl       0.98      1.00      0.99       500
          pl       1.00      1.00      1.00       500
          pt       1.00      1.00      1.00       500
          ru       1.00      1.00      1.00       500
          sw       0.95      1.00      0.98       500
          th       1.00      1.00      1.00       500
  

In [26]:
y_test_pred = best_model.predict(X_test)

print("TEST Accuracy:", accuracy_score(y_test, y_test_pred))
print("TEST F1 Macro:", f1_score(y_test, y_test_pred, average="macro"))

TEST Accuracy: 0.995
TEST F1 Macro: 0.9950108361151926


---


### Save Model

In [31]:
import joblib

joblib.dump(best_model, "svc_language_detector_v2.pkl")

['svc_language_detector_v2.pkl']

### Load and Predict

In [32]:
model = joblib.load("svc_language_detector_v2.pkl")

In [33]:
text = ["كيف الحال ؟"]
model.predict(text)

array(['ar'], dtype=object)

---